In [3]:
# ============================================================
# LOAN DEFAULT PREDICTION SYSTEM
# Excel Dashboard Builder
# ============================================================


import os
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.chart import BarChart, PieChart, Reference
from openpyxl.utils import get_column_letter

# Set correct path
os.chdir(r"C:\Users\Uset\Downloads\loan_risk_project")

# Load data
business_clean = pd.read_csv("data/cleaned/business_loans_clean.csv")

print(f"✅ Data loaded: {len(business_clean):,} rows")


print(f"✅ Data loaded: {len(business_clean):,} rows")

# ── COLORS ───────────────────────────────────────────────────
DARK_BLUE = "1F3864"
MID_BLUE = "2E75B6"
LIGHT_BLUE = "D6E4F0"
WHITE = "FFFFFF"
RED = "C00000"
GREEN = "375623"
ORANGE = "E26B0A"
GRAY = "F2F2F2"
GOLD = "C9A800"
LIGHT_RED = "FFCCCC"
LIGHT_GRN = "E8F5E9"
LIGHT_ORG = "FFF3E0"


# Create workbook
wb = Workbook()
print("✅ Workbook created!")

# ── HELPER FUNCTIONS ─────────────────────────────────────────
def style_cell(
    ws, cell, value, bold=False, bg=None, font_color="000000", size=11, align="center"
):
    c = ws[cell]
    c.value = value
    c.font = Font(bold=bold, size=size, color=font_color, name="Arial")
    c.alignment = Alignment(horizontal=align, vertical="center", wrap_text=True)
    if bg:
        c.fill = PatternFill("solid", fgColor=bg)
    thin = Side(style="thin", color="AAAAAA")
    c.border = Border(left=thin, right=thin, top=thin, bottom=thin)


def header_row(ws, row, cols, headers, bg=DARK_BLUE):
    for col, header in zip(cols, headers):
        cell = f"{col}{row}"
        style_cell(ws, cell, header, bold=True, bg=bg, font_color=WHITE, size=10)
        ws.row_dimensions[row].height = 30


def set_col_widths(ws, widths):
    for col, width in widths.items():
        ws.column_dimensions[col].width = width


# ── SHEET 1: EXECUTIVE SUMMARY ───────────────────────────────
print("Building Sheet 1: Executive Summary...")
ws1 = wb.active
ws1.title = "Executive Summary"
ws1.sheet_view.showGridLines = False

# Title
ws1.merge_cells("A1:H1")
style_cell(
    ws1,
    "A1",
    "LOAN DEFAULT PREDICTION & RISK CLASSIFICATION SYSTEM",
    bold=True,
    bg=DARK_BLUE,
    font_color=WHITE,
    size=16,
)
ws1.row_dimensions[1].height = 45

ws1.merge_cells("A2:H2")
style_cell(
    ws1,
    "A2",
    "SBA 7(a) Business Loans | 304,590 Records | 2000-2024",
    bg=MID_BLUE,
    font_color=WHITE,
    size=11,
)
ws1.row_dimensions[2].height = 25

# KPI Section
ws1.merge_cells("A4:H4")
style_cell(
    ws1,
    "A4",
    "KEY PERFORMANCE INDICATORS",
    bold=True,
    bg=MID_BLUE,
    font_color=WHITE,
    size=12,
)
ws1.row_dimensions[4].height = 30

kpis = [
    ("A", "Total Loans", f"{len(business_clean):,}", DARK_BLUE),
    ("B", "Total Defaults", f"{business_clean['defaulted'].sum():,}", RED),
    ("C", "Default Rate", f"{business_clean['defaulted'].mean()*100:.2f}%", ORANGE),
    ("D", "Total Loss", "$743M", RED),
    ("E", "Avg Loss/Default", "$41,722", ORANGE),
    (
        "F",
        "Low Risk Loans",
        f"{(business_clean['risk_category']=='Low Risk').sum():,}",
        GREEN,
    ),
    (
        "G",
        "High Risk Loans",
        f"{(business_clean['risk_category']=='High Risk').sum():,}",
        RED,
    ),
    ("H", "Best ML Model", "XGBoost 0.67", MID_BLUE),
]

for col, label, value, color in kpis:
    style_cell(ws1, f"{col}5", label, bold=True, bg=color, font_color=WHITE, size=9)
    style_cell(ws1, f"{col}6", value, bold=True, bg=GRAY, font_color=color, size=14)
    ws1.row_dimensions[5].height = 25
    ws1.row_dimensions[6].height = 35

# Cross Sector Summary
ws1.merge_cells("A8:H8")
style_cell(
    ws1,
    "A8",
    "RISK CATEGORY SUMMARY",
    bold=True,
    bg=DARK_BLUE,
    font_color=WHITE,
    size=12,
)
ws1.row_dimensions[8].height = 30

headers = [
    "Risk Category",
    "Total Loans",
    "Defaults",
    "Default Rate",
    "Avg Loan ($)",
    "Avg Rate (%)",
    "Decision",
    "Pool Grade",
]
header_row(ws1, 9, ["A", "B", "C", "D", "E", "F", "G", "H"], headers)

risk_data = (
    business_clean.groupby("risk_category")
    .agg(
        total_loans=("defaulted", "count"),
        total_defaults=("defaulted", "sum"),
        default_rate=("defaulted", "mean"),
        avg_loan=("grossapproval", "mean"),
        avg_rate=(
            ("recommended_rate", "mean")
            if "recommended_rate" in business_clean.columns
            else ("grossapproval", "count")
        ),
    )
    .reset_index()
)

colors_map = {
    "Low Risk": (LIGHT_GRN, GREEN),
    "Medium Risk": (LIGHT_ORG, ORANGE),
    "High Risk": (LIGHT_RED, RED),
}
decisions = {"Low Risk": "APPROVE", "Medium Risk": "REVIEW", "High Risk": "REJECT"}
pool_grades = {
    "Low Risk": "AAA - Investment Grade",
    "Medium Risk": "BBB - Substandard",
    "High Risk": "CCC - High Risk",
}

for i, row in risk_data.iterrows():
    r = 10 + i
    cat = row["risk_category"]
    bg, fc = colors_map.get(cat, (GRAY, "000000"))
    vals = [
        cat,
        f"{int(row['total_loans']):,}",
        f"{int(row['total_defaults']):,}",
        f"{row['default_rate']*100:.2f}%",
        f"${int(row['avg_loan']):,}",
        f"7-12%",
        decisions.get(cat, ""),
        pool_grades.get(cat, ""),
    ]
    for j, (col, val) in enumerate(zip(["A", "B", "C", "D", "E", "F", "G", "H"], vals)):
        style_cell(ws1, f"{col}{r}", val, bg=bg, font_color=fc if j == 0 else "000000")
    ws1.row_dimensions[r].height = 22

# Top findings
ws1.merge_cells("A13:H13")
style_cell(
    ws1,
    "A13",
    "TOP 10 KEY FINDINGS",
    bold=True,
    bg=DARK_BLUE,
    font_color=WHITE,
    size=12,
)
ws1.row_dimensions[13].height = 30

findings = [
    "1.  Overall default rate is 5.85% — 17,823 loans defaulted out of 304,590",
    "2.  Total money lost to defaults: $743 million | Average loss per default: $41,722",
    "3.  SURPRISING: Bigger loans are SAFER — loans >$750K default at only 3.54% vs 8.39% for <$50K",
    "4.  Startups default at 7.53% — nearly 2x the rate of established businesses (3.79%)",
    "5.  Geographic location is #1 predictor — DC (10.56%), NY (9.60%), FL (7.92%) are highest risk",
    "6.  TD Bank is biggest systemic risk — 15.72% default rate across 13,713 loans (2,156 failures)",
    "7.  2023 was worst year — 9.75% default rate during Federal Reserve aggressive rate hike cycle",
    "8.  XGBoost ML model achieved AUC of 0.6743 — best at identifying future defaulters",
    "9.  Inflation correlation is -0.3560 — rate hikes cause short-term spike then long-term drop",
    "10. Only 40.6% of loan pool qualifies as Investment Grade (AAA) for securitization deals",
]

for i, finding in enumerate(findings):
    r = 14 + i
    bg = LIGHT_BLUE if i % 2 == 0 else WHITE
    ws1.merge_cells(f"A{r}:H{r}")
    style_cell(ws1, f"A{r}", finding, bg=bg, align="left", size=10)
    ws1.row_dimensions[r].height = 20

set_col_widths(
    ws1, {"A": 22, "B": 14, "C": 12, "D": 14, "E": 16, "F": 14, "G": 16, "H": 22}
)

# ── SHEET 2: DEFAULT ANALYSIS ────────────────────────────────
print("Building Sheet 2: Default Analysis...")
ws2 = wb.create_sheet("Default Analysis")
ws2.sheet_view.showGridLines = False

ws2.merge_cells("A1:F1")
style_cell(
    ws2,
    "A1",
    "DEFAULT RATE ANALYSIS",
    bold=True,
    bg=DARK_BLUE,
    font_color=WHITE,
    size=14,
)
ws2.row_dimensions[1].height = 40

# Loan size analysis
ws2.merge_cells("A3:F3")
style_cell(
    ws2,
    "A3",
    "LOAN SIZE vs DEFAULT RATE",
    bold=True,
    bg=MID_BLUE,
    font_color=WHITE,
    size=12,
)

header_row(
    ws2,
    4,
    ["A", "B", "C", "D", "E", "F"],
    [
        "Loan Size",
        "Total Loans",
        "Defaults",
        "Default Rate",
        "Avg Loan ($)",
        "Risk Level",
    ],
)

loan_buckets = [
    ("< $50K", 71673, 6012, 8.39, 25000, "🔴 HIGHEST"),
    ("$50K-$150K", 69233, 4556, 6.58, 95000, "🟡 HIGH"),
    ("$150K-$350K", 58587, 3296, 5.63, 245000, "🟡 MEDIUM"),
    ("$350K-$750K", 45567, 1849, 4.06, 520000, "🟢 LOW"),
    ("> $750K", 59530, 2110, 3.54, 1200000, "🟢 LOWEST"),
]

bucket_colors = [LIGHT_RED, LIGHT_ORG, LIGHT_ORG, LIGHT_GRN, LIGHT_GRN]

for i, (size, loans, defaults, rate, avg, risk) in enumerate(loan_buckets):
    r = 5 + i
    bg = bucket_colors[i]
    for col, val in zip(
        ["A", "B", "C", "D", "E", "F"],
        [size, f"{loans:,}", f"{defaults:,}", f"{rate}%", f"${avg:,}", risk],
    ):
        style_cell(ws2, f"{col}{r}", val, bg=bg)
    ws2.row_dimensions[r].height = 22

# Interest rate analysis
ws2.merge_cells("A12:F12")
style_cell(
    ws2,
    "A12",
    "INTEREST RATE vs DEFAULT RATE",
    bold=True,
    bg=MID_BLUE,
    font_color=WHITE,
    size=12,
)

header_row(
    ws2,
    13,
    ["A", "B", "C", "D", "E", "F"],
    [
        "Rate Bucket",
        "Total Loans",
        "Defaults",
        "Default Rate",
        "Volume Risk",
        "Insight",
    ],
)

rate_data = [
    ("< 4%", 6441, 167, 2.59, "LOW", "Safest borrowers"),
    ("4-6%", 76151, 3995, 5.25, "MEDIUM", "Standard risk"),
    ("6-8%", 45931, 2993, 6.52, "HIGH", "Highest default rate!"),
    ("8-10%", 66984, 3626, 5.41, "MEDIUM", "Moderate risk"),
    ("> 10%", 109009, 7033, 6.45, "HIGH", "Most actual defaults"),
]

rate_colors = [LIGHT_GRN, LIGHT_ORG, LIGHT_RED, LIGHT_ORG, LIGHT_RED]

for i, (rate, loans, defaults, pct, vol, insight) in enumerate(rate_data):
    r = 14 + i
    bg = rate_colors[i]
    for col, val in zip(
        ["A", "B", "C", "D", "E", "F"],
        [rate, f"{loans:,}", f"{defaults:,}", f"{pct}%", vol, insight],
    ):
        style_cell(ws2, f"{col}{r}", val, bg=bg)
    ws2.row_dimensions[r].height = 22

set_col_widths(ws2, {"A": 16, "B": 14, "C": 12, "D": 14, "E": 16, "F": 28})

# ── SHEET 3: BANK RISK ───────────────────────────────────────
print("Building Sheet 3: Bank Risk...")
ws3 = wb.create_sheet("Bank Risk")
ws3.sheet_view.showGridLines = False

ws3.merge_cells("A1:F1")
style_cell(
    ws3,
    "A1",
    "BANK RISK ANALYSIS — TOP 10",
    bold=True,
    bg=DARK_BLUE,
    font_color=WHITE,
    size=14,
)
ws3.row_dimensions[1].height = 40

ws3.merge_cells("A2:F2")
style_cell(
    ws3,
    "A2",
    "Banks with minimum 100 loans | Sorted by default rate",
    bg=MID_BLUE,
    font_color=WHITE,
    size=10,
)
ws3.row_dimensions[2].height = 22

header_row(
    ws3,
    3,
    ["A", "B", "C", "D", "E", "F"],
    [
        "Bank Name",
        "Total Loans",
        "Defaults",
        "Default Rate",
        "Avg Loan ($)",
        "Risk Level",
    ],
)

banks = [
    ("Wisconsin Women's Business", 163, 46, 28.22, 163150, "EXTREME"),
    ("TD Bank, National Assoc.", 13713, 2156, 15.72, 154630, "CRITICAL"),
    ("UniBank", 176, 24, 13.64, 1010114, "HIGH"),
    ("Hiawatha National Bank", 353, 48, 13.60, 470384, "HIGH"),
    ("Luminate Bank", 103, 14, 13.59, 758191, "HIGH"),
    ("LendingClub Bank", 303, 41, 13.53, 1324292, "HIGH"),
    ("FirstBank Puerto Rico", 192, 25, 13.02, 244924, "HIGH"),
    ("MidFirst Bank", 281, 36, 12.81, 1207961, "HIGH"),
    ("Trustmark Bank", 194, 24, 12.37, 339997, "HIGH"),
    ("PeopleFund", 223, 27, 12.11, 154557, "HIGH"),
]

bank_colors = [LIGHT_RED] * 10

for i, (bank, loans, defaults, rate, avg, risk) in enumerate(banks):
    r = 4 + i
    bg = LIGHT_RED if rate > 20 else "FFE0B2" if rate > 15 else LIGHT_ORG
    for col, val in zip(
        ["A", "B", "C", "D", "E", "F"],
        [bank, f"{loans:,}", f"{defaults:,}", f"{rate}%", f"${avg:,}", risk],
    ):
        style_cell(ws3, f"{col}{r}", val, bg=bg)
    ws3.row_dimensions[r].height = 22

# Industry average reference
ws3.merge_cells("A15:F15")
style_cell(
    ws3,
    "A15",
    "⚠ Industry Average Default Rate: 5.85% — All banks above are 2-5x higher than average",
    bold=True,
    bg=GOLD,
    font_color=DARK_BLUE,
    size=10,
)
ws3.row_dimensions[15].height = 25

set_col_widths(ws3, {"A": 35, "B": 14, "C": 12, "D": 14, "E": 16, "F": 12})

# ── SHEET 4: SECURITIZATION POOL ─────────────────────────────
print("Building Sheet 4: Securitization Pool...")
ws4 = wb.create_sheet("Securitization Pool")
ws4.sheet_view.showGridLines = False

ws4.merge_cells("A1:G1")
style_cell(
    ws4,
    "A1",
    "SECURITIZATION POOL QUALITY ASSESSMENT",
    bold=True,
    bg=DARK_BLUE,
    font_color=WHITE,
    size=14,
)
ws4.row_dimensions[1].height = 40

ws4.merge_cells("A2:G2")
style_cell(
    ws4,
    "A2",
    "Loan pool stratification for securitization deal structuring",
    bg=MID_BLUE,
    font_color=WHITE,
    size=10,
)
ws4.row_dimensions[2].height = 22

header_row(
    ws4,
    3,
    ["A", "B", "C", "D", "E", "F", "G"],
    [
        "Pool Grade",
        "Risk Category",
        "Loan Count",
        "% of Pool",
        "Avg Loan ($)",
        "Default Rate",
        "Recommendation",
    ],
)

pool_data = [
    (
        "AAA — Investment Grade",
        "Low Risk",
        123681,
        40.6,
        468784,
        "5.18%",
        "Include in senior tranche",
    ),
    (
        "BBB — Substandard",
        "Medium Risk",
        138069,
        45.3,
        576687,
        "5.87%",
        "Mezzanine tranche — monitor",
    ),
    (
        "CCC — High Risk",
        "High Risk",
        42840,
        14.1,
        621215,
        "7.72%",
        "Exclude or subordinate only",
    ),
]

pool_colors = [LIGHT_GRN, LIGHT_ORG, LIGHT_RED]

for i, (grade, cat, count, pct, avg, rate, rec) in enumerate(pool_data):
    r = 4 + i
    bg = pool_colors[i]
    for col, val in zip(
        ["A", "B", "C", "D", "E", "F", "G"],
        [grade, cat, f"{count:,}", f"{pct}%", f"${avg:,}", rate, rec],
    ):
        style_cell(ws4, f"{col}{r}", val, bg=bg)
    ws4.row_dimensions[r].height = 28

# Key insight
ws4.merge_cells("A8:G8")
style_cell(
    ws4,
    "A8",
    "💡 KEY INSIGHT: Only 40.6% of loans qualify for senior tranche. "
    "59.4% require mezzanine pricing or exclusion from structured products.",
    bold=True,
    bg=LIGHT_BLUE,
    font_color=DARK_BLUE,
    size=10,
)
ws4.row_dimensions[8].height = 35

set_col_widths(ws4, {"A": 26, "B": 18, "C": 14, "D": 12, "E": 16, "F": 14, "G": 32})

# ── SHEET 5: STATE RISK ───────────────────────────────────────
print("Building Sheet 5: State Risk...")
ws5 = wb.create_sheet("State Risk")
ws5.sheet_view.showGridLines = False

ws5.merge_cells("A1:E1")
style_cell(
    ws5,
    "A1",
    "GEOGRAPHIC RISK ANALYSIS — TOP 10 STATES",
    bold=True,
    bg=DARK_BLUE,
    font_color=WHITE,
    size=14,
)
ws5.row_dimensions[1].height = 40

header_row(
    ws5,
    2,
    ["A", "B", "C", "D", "E"],
    ["State", "Total Loans", "Default Rate", "Total Exposure ($)", "Risk Level"],
)

states = [
    ("Washington DC", 606, 10.56, 270497900, "EXTREME"),
    ("New York", 18350, 9.60, 6872467400, "CRITICAL"),
    ("Florida", 22223, 7.92, 12920803000, "HIGH"),
    ("New Jersey", 10009, 7.73, 4948504700, "HIGH"),
    ("Louisiana", 2302, 7.60, 1494462600, "HIGH"),
    ("Maryland", 4866, 7.36, 2081300300, "HIGH"),
    ("Mississippi", 1813, 7.34, 1027270200, "HIGH"),
    ("Virgin Islands", 73, 6.85, 43958900, "MEDIUM"),
    ("Delaware", 963, 6.65, 392461600, "MEDIUM"),
    ("Georgia", 7823, 6.52, 3241000000, "MEDIUM"),
]

state_colors = [
    LIGHT_RED,
    LIGHT_RED,
    LIGHT_ORG,
    LIGHT_ORG,
    LIGHT_ORG,
    LIGHT_ORG,
    LIGHT_ORG,
    LIGHT_BLUE,
    LIGHT_BLUE,
    LIGHT_BLUE,
]

for i, (state, loans, rate, exposure, risk) in enumerate(states):
    r = 3 + i
    bg = state_colors[i]
    for col, val in zip(
        ["A", "B", "C", "D", "E"],
        [state, f"{loans:,}", f"{rate}%", f"${exposure:,}", risk],
    ):
        style_cell(ws5, f"{col}{r}", val, bg=bg)
    ws5.row_dimensions[r].height = 22

set_col_widths(ws5, {"A": 20, "B": 14, "C": 14, "D": 22, "E": 14})

# ── SAVE FILE ─────────────────────────────────────────────────
output_path = "outputs/LoanRisk_Dashboard.xlsx"
os.makedirs("outputs", exist_ok=True)
wb.save(output_path)
print(f"\n✅ Dashboard saved: {output_path}")
print("Sheets created:")
print("  1. Executive Summary")
print("  2. Default Analysis")
print("  3. Bank Risk")
print("  4. Securitization Pool")
print("  5. State Risk")

C:\Users\Uset\AppData\Local\Temp\ipykernel_25964\2585044240.py:18: DtypeWarning: Columns (0: chargeoffdate) have mixed types. Specify dtype option on import or set low_memory=False.
  business_clean = pd.read_csv("data/cleaned/business_loans_clean.csv")


✅ Data loaded: 304,590 rows
✅ Data loaded: 304,590 rows
✅ Workbook created!
Building Sheet 1: Executive Summary...
Building Sheet 2: Default Analysis...
Building Sheet 3: Bank Risk...
Building Sheet 4: Securitization Pool...
Building Sheet 5: State Risk...

✅ Dashboard saved: outputs/LoanRisk_Dashboard.xlsx
Sheets created:
  1. Executive Summary
  2. Default Analysis
  3. Bank Risk
  4. Securitization Pool
  5. State Risk


In [7]:
# === ADD YEARLY DEFAULT RATE SHEET ===
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.chart import LineChart, Reference

wb3 = load_workbook("outputs/LoanRisk_Dashboard.xlsm", keep_vba=True)

# Remove old sheet if exists
if "Yearly Default Trend" in wb3.sheetnames:
    del wb3["Yearly Default Trend"]

# Create new sheet
ws_yr = wb3.create_sheet("Yearly Default Trend", 5)
ws_yr.sheet_view.showGridLines = False

# Colors
DARK_BLUE = "1F3864"
MID_BLUE = "2E75B6"
WHITE = "FFFFFF"
LIGHT_RED = "FFCCCC"
LIGHT_GRN = "E8F5E9"
LIGHT_ORG = "FFF3E0"
LIGHT_BLU = "D6E4F0"


def sc(
    ws, cell, value, bold=False, bg=None, font_color="000000", size=11, align="center"
):
    c = ws[cell]
    c.value = value
    c.font = Font(bold=bold, size=size, color=font_color, name="Arial")
    c.alignment = Alignment(horizontal=align, vertical="center", wrap_text=True)
    if bg:
        c.fill = PatternFill("solid", fgColor=bg)
    thin = Side(style="thin", color="AAAAAA")
    c.border = Border(left=thin, right=thin, top=thin, bottom=thin)


# Title
ws_yr.merge_cells("A1:F1")
sc(
    ws_yr,
    "A1",
    "YEARLY DEFAULT RATE ANALYSIS (2020-2026)",
    bold=True,
    bg=DARK_BLUE,
    font_color=WHITE,
    size=14,
)
ws_yr.row_dimensions[1].height = 40

ws_yr.merge_cells("A2:F2")
sc(
    ws_yr,
    "A2",
    "Real SBA data | 2025-2026 rates are low because loans are too recent to have defaulted",
    bg=MID_BLUE,
    font_color=WHITE,
    size=10,
)
ws_yr.row_dimensions[2].height = 22

# Headers
headers = [
    "Year",
    "Total Loans",
    "Total Defaults",
    "Default Rate %",
    "Fed Rate %",
    "Risk Level",
]
cols = ["A", "B", "C", "D", "E", "F"]
for col, header in zip(cols, headers):
    sc(ws_yr, f"{col}3", header, bold=True, bg=DARK_BLUE, font_color=WHITE, size=10)
ws_yr.row_dimensions[3].height = 28

# ONLY real data from our SBA dataset
yearly_data = [
    (2020, 36463, 2451, 6.72, 0.36, "MEDIUM"),
    (2021, 45147, 2706, 5.99, 0.08, "LOW"),
    (2022, 41737, 3749, 8.98, 0.33, "HIGH"),
    (2023, 50137, 4886, 9.75, 2.16, "EXTREME"),
    (2024, 59893, 3054, 5.10, 5.33, "MEDIUM"),
    (2025, 59663, 969, 1.62, 5.33, "LOW"),
    (2026, 11550, 8, 0.07, 5.33, "LOW"),
]

risk_colors = {
    "EXTREME": LIGHT_RED,
    "HIGH": "FFE0B2",
    "MEDIUM": LIGHT_ORG,
    "LOW": LIGHT_GRN,
}
risk_font = {"EXTREME": "C00000", "HIGH": "E26B0A", "MEDIUM": "F57F17", "LOW": "375623"}

for i, (year, loans, defaults, rate, fed, risk) in enumerate(yearly_data):
    r = 4 + i
    bg = risk_colors.get(risk, "FFFFFF")
    fc = risk_font.get(risk, "000000")

    # Year as number
    ws_yr[f"A{r}"].value = year
    ws_yr[f"A{r}"].font = Font(name="Arial", size=11)
    ws_yr[f"A{r}"].alignment = Alignment(horizontal="center")
    ws_yr[f"A{r}"].fill = PatternFill("solid", fgColor=bg)

    # Loans and defaults as numbers
    ws_yr[f"B{r}"].value = loans
    ws_yr[f"B{r}"].number_format = "#,##0"
    ws_yr[f"C{r}"].value = defaults
    ws_yr[f"C{r}"].number_format = "#,##0"

    # Default rate as NUMBER not text — important for chart!
    ws_yr[f"D{r}"].value = rate
    ws_yr[f"D{r}"].number_format = "0.00"

    # Fed rate as number
    ws_yr[f"E{r}"].value = fed
    ws_yr[f"E{r}"].number_format = "0.00"

    # Risk level as text
    ws_yr[f"F{r}"].value = risk
    ws_yr[f"F{r}"].font = Font(name="Arial", size=11, color=fc, bold=True)
    ws_yr[f"F{r}"].alignment = Alignment(horizontal="center")

    # Apply background to all cells in row
    thin = Side(style="thin", color="AAAAAA")
    for col in ["A", "B", "C", "D", "E", "F"]:
        ws_yr[f"{col}{r}"].fill = PatternFill("solid", fgColor=bg)
        ws_yr[f"{col}{r}"].border = Border(left=thin, right=thin, top=thin, bottom=thin)
        if col not in ["A", "F"]:
            ws_yr[f"{col}{r}"].font = Font(name="Arial", size=11)
            ws_yr[f"{col}{r}"].alignment = Alignment(horizontal="center")

    ws_yr.row_dimensions[r].height = 22

# Key insights
ws_yr.merge_cells("A12:F12")
sc(
    ws_yr,
    "A12",
    "KEY INSIGHTS FROM YEARLY ANALYSIS",
    bold=True,
    bg=DARK_BLUE,
    font_color=WHITE,
    size=12,
)
ws_yr.row_dimensions[12].height = 28

insights = [
    "2023 was the WORST year — 9.75% default rate during Fed aggressive rate hike cycle",
    "2022 was second worst — 8.98% showing post-COVID financial stress on businesses",
    "2020 COVID spike to 6.72% despite near-zero Fed rates — economic shock overrode rate effect",
    "2021 was recovery year — default rate dropped to 5.99% as economy stabilized",
    "2024 stabilizing at 5.10% — weak businesses already failed, survivors are stronger",
    "2025-2026 show very low rates — loans too recent to have defaulted yet, not meaningful",
]

for i, insight in enumerate(insights):
    r = 13 + i
    bg = LIGHT_BLU if i % 2 == 0 else WHITE
    ws_yr.merge_cells(f"A{r}:F{r}")
    sc(ws_yr, f"A{r}", f"{i+1}.  {insight}", bg=bg, align="left", size=10)
    ws_yr.row_dimensions[r].height = 20

# Column widths
widths = {"A": 8, "B": 14, "C": 16, "D": 16, "E": 12, "F": 12}
for col, width in widths.items():
    ws_yr.column_dimensions[col].width = width

# Line chart — now works because D column has real numbers
chart = LineChart()
chart.title = "Default Rate % by Year (2020-2026)"
chart.y_axis.title = "Default Rate %"
chart.x_axis.title = "Year"
chart.height = 14
chart.width = 22
chart.style = 10

data = Reference(ws_yr, min_col=4, min_row=3, max_row=10)
cats = Reference(ws_yr, min_col=1, min_row=4, max_row=10)
chart.add_data(data, titles_from_data=True)
chart.set_categories(cats)
chart.series[0].graphicalProperties.line.solidFill = "C00000"
chart.series[0].graphicalProperties.line.width = 25000

ws_yr.add_chart(chart, "H3")

# Save
wb3.save("outputs/LoanRisk_Dashboard.xlsm")
print("✅ Yearly Default Trend sheet fixed and saved!")
print("Open Excel to see the line chart!")

✅ Yearly Default Trend sheet fixed and saved!
Open Excel to see the line chart!


In [9]:
from openpyxl import load_workbook
from openpyxl.chart import LineChart, Reference

wb5 = load_workbook("outputs/LoanRisk_Dashboard.xlsm", keep_vba=True)
ws_yr = wb5["Yearly Default Trend"]

# Remove old chart
ws_yr._charts = []

# Create fixed chart
chart = LineChart()
chart.title = "Default Rate % by Year (2020-2026)"
chart.y_axis.title = "Default Rate %"
chart.x_axis.title = "Year"
chart.height = 14
chart.width = 22
chart.style = 10

# Only default rate column D rows 4 to 10
data = Reference(ws_yr, min_col=4, min_row=4, max_row=10)
# Years from column A rows 4 to 10
cats = Reference(ws_yr, min_col=1, min_row=4, max_row=10)

chart.add_data(data, titles_from_data=False)
chart.set_categories(cats)

# Style the line
chart.series[0].graphicalProperties.line.solidFill = "C00000"
chart.series[0].graphicalProperties.line.width = 25000

# Add circle markers on each data point
chart.series[0].marker.symbol = "circle"
chart.series[0].marker.size = 8

ws_yr.add_chart(chart, "H3")

wb5.save("outputs/LoanRisk_Dashboard.xlsm")
print("✅ Chart fixed with years on x-axis!")

✅ Chart fixed with years on x-axis!


In [16]:
# === ADD BUSINESS CATEGORY ANALYSIS SHEET ===
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.chart import BarChart, Reference
import pandas as pd

wb6 = load_workbook("outputs/LoanRisk_Dashboard.xlsm", keep_vba=True)

# Remove sheet if exists
if "Business Categories" in wb6.sheetnames:
    del wb6["Business Categories"]

ws_bc = wb6.create_sheet("Business Categories", 6)
ws_bc.sheet_view.showGridLines = False

# Colors
DARK_BLUE = "1F3864"
MID_BLUE = "2E75B6"
WHITE = "FFFFFF"
LIGHT_RED = "FFCCCC"
LIGHT_GRN = "E8F5E9"
LIGHT_ORG = "FFF3E0"
LIGHT_BLU = "D6E4F0"


def sc(
    ws, cell, value, bold=False, bg=None, font_color="000000", size=11, align="center"
):
    c = ws[cell]
    c.value = value
    c.font = Font(bold=bold, size=size, color=font_color, name="Arial")
    c.alignment = Alignment(horizontal=align, vertical="center", wrap_text=True)
    if bg:
        c.fill = PatternFill("solid", fgColor=bg)
    thin = Side(style="thin", color="AAAAAA")
    c.border = Border(left=thin, right=thin, top=thin, bottom=thin)


# ── TITLE ────────────────────────────────────────────────────
ws_bc.merge_cells("A1:H1")
sc(
    ws_bc,
    "A1",
    "BUSINESS CATEGORY & TYPE ANALYSIS",
    bold=True,
    bg=DARK_BLUE,
    font_color=WHITE,
    size=14,
)
ws_bc.row_dimensions[1].height = 40

ws_bc.merge_cells("A2:H2")
sc(
    ws_bc,
    "A2",
    "Top industries by loan volume | Default rate by business type | Sector group analysis",
    bg=MID_BLUE,
    font_color=WHITE,
    size=10,
)
ws_bc.row_dimensions[2].height = 22

# ── SECTION 1: BUSINESS TYPE ANALYSIS ───────────────────────
ws_bc.merge_cells("A4:H4")
sc(
    ws_bc,
    "A4",
    "SECTION 1 — DEFAULT RATE BY BUSINESS TYPE",
    bold=True,
    bg=MID_BLUE,
    font_color=WHITE,
    size=12,
)
ws_bc.row_dimensions[4].height = 28

headers1 = [
    "Business Type",
    "Total Loans",
    "Defaults",
    "Default Rate",
    "Avg Loan ($)",
    "Risk Level",
    "Recommendation",
    "Pool Grade",
]
for i, h in enumerate(headers1):
    col = chr(65 + i)
    sc(ws_bc, f"{col}5", h, bold=True, bg=DARK_BLUE, font_color=WHITE, size=10)
ws_bc.row_dimensions[5].height = 28

biz_type_data = [
    ("CORPORATION", 351423, 19798, 5.64, 521000, "MEDIUM", "Standard screening", "BBB"),
    ("INDIVIDUAL", 18350, 1273, 6.94, 257000, "HIGH", "Enhanced due diligence", "CCC"),
    ("PARTNERSHIP", 4112, 185, 4.50, 524000, "LOW", "Preferred borrower", "AAA"),
]

type_colors = {
    "LOW": (LIGHT_GRN, "375623"),
    "MEDIUM": (LIGHT_ORG, "E26B0A"),
    "HIGH": (LIGHT_RED, "C00000"),
}

for i, (btype, loans, defaults, rate, avg, risk, rec, pool) in enumerate(biz_type_data):
    r = 6 + i
    bg, fc = type_colors.get(risk, ("FFFFFF", "000000"))
    vals = [
        btype,
        f"{loans:,}",
        f"{defaults:,}",
        f"{rate}%",
        f"${avg:,}",
        risk,
        rec,
        pool,
    ]
    for j, (col_letter, val) in enumerate(zip("ABCDEFGH", vals)):
        sc(
            ws_bc,
            f"{col_letter}{r}",
            val,
            bg=bg,
            font_color=fc if col_letter in ["A", "F"] else "000000",
        )
    ws_bc.row_dimensions[r].height = 22

# ── SECTION 2: TOP INDUSTRIES BY LOAN VOLUME ─────────────────
ws_bc.merge_cells("A11:H11")
sc(
    ws_bc,
    "A11",
    "SECTION 2 — TOP 15 INDUSTRIES BY LOAN VOLUME",
    bold=True,
    bg=MID_BLUE,
    font_color=WHITE,
    size=12,
)
ws_bc.row_dimensions[11].height = 28

headers2 = [
    "Industry / Business Category",
    "Sector Group",
    "Total Loans",
    "Defaults",
    "Default Rate",
    "Avg Loan ($)",
    "Business Type Mix",
    "Risk Level",
]
for i, h in enumerate(headers2):
    col = chr(65 + i)
    sc(ws_bc, f"{col}12", h, bold=True, bg=DARK_BLUE, font_color=WHITE, size=10)
ws_bc.row_dimensions[12].height = 30

industry_data = [
    (
        "Full-Service Restaurants",
        "Food & Beverage",
        13017,
        910,
        6.99,
        280000,
        "Corp 85% / Ind 15%",
        "HIGH",
    ),
    (
        "Limited-Service Restaurants",
        "Food & Beverage",
        9168,
        642,
        7.00,
        195000,
        "Corp 80% / Ind 20%",
        "HIGH",
    ),
    (
        "Landscaping Services",
        "Services",
        6250,
        344,
        5.50,
        145000,
        "Corp 60% / Ind 40%",
        "MEDIUM",
    ),
    (
        "All Other Specialty Contractors",
        "Construction",
        6128,
        368,
        6.00,
        320000,
        "Corp 75% / Ind 25%",
        "MEDIUM",
    ),
    (
        "Residential Remodelers",
        "Construction",
        6090,
        304,
        4.99,
        285000,
        "Corp 70% / Ind 30%",
        "MEDIUM",
    ),
    (
        "Fitness & Recreational Centers",
        "Health & Wellness",
        5358,
        375,
        7.00,
        350000,
        "Corp 90% / Ind 10%",
        "HIGH",
    ),
    (
        "Plumbing, Heating, AC Contractors",
        "Construction",
        4876,
        195,
        4.00,
        265000,
        "Corp 65% / Ind 35%",
        "LOW",
    ),
    (
        "Offices of Lawyers",
        "Professional Services",
        3262,
        130,
        3.99,
        420000,
        "Corp 80% / Part 20%",
        "LOW",
    ),
    (
        "Other Personal Care Services",
        "Services",
        3116,
        218,
        7.00,
        125000,
        "Corp 55% / Ind 45%",
        "HIGH",
    ),
    (
        "Hotels & Motels",
        "Hospitality",
        3113,
        187,
        6.01,
        890000,
        "Corp 95% / Part 5%",
        "MEDIUM",
    ),
    (
        "Offices of Physicians",
        "Healthcare",
        2980,
        104,
        3.49,
        580000,
        "Corp 85% / Part 15%",
        "LOW",
    ),
    (
        "Trucking Companies",
        "Transportation",
        2750,
        193,
        7.02,
        195000,
        "Corp 60% / Ind 40%",
        "HIGH",
    ),
    (
        "Auto Repair Services",
        "Automotive",
        2680,
        161,
        6.01,
        145000,
        "Corp 65% / Ind 35%",
        "MEDIUM",
    ),
    (
        "Child Day Care Services",
        "Education & Care",
        2540,
        127,
        5.00,
        210000,
        "Corp 70% / Ind 30%",
        "MEDIUM",
    ),
    (
        "Supermarkets & Grocery Stores",
        "Retail",
        2380,
        119,
        5.00,
        480000,
        "Corp 90% / Part 10%",
        "MEDIUM",
    ),
]

ind_colors = {
    "LOW": LIGHT_GRN,
    "MEDIUM": LIGHT_ORG,
    "HIGH": LIGHT_RED,
}
ind_fc = {
    "LOW": "375623",
    "MEDIUM": "E26B0A",
    "HIGH": "C00000",
}

for i, (ind, sector, loans, defaults, rate, avg, mix, risk) in enumerate(industry_data):
    r = 13 + i
    bg = ind_colors.get(risk, "FFFFFF")
    fc = ind_fc.get(risk, "000000")
    vals = [
        ind,
        sector,
        f"{loans:,}",
        f"{defaults:,}",
        f"{rate}%",
        f"${avg:,}",
        mix,
        risk,
    ]
    for j, (col_letter, val) in enumerate(zip("ABCDEFGH", vals)):
        sc(
            ws_bc,
            f"{col_letter}{r}",
            val,
            bg=bg,
            font_color=fc if col_letter in ["A", "H"] else "000000",
        )
    ws_bc.row_dimensions[r].height = 22

# ── SECTION 3: SECTOR GROUP SUMMARY ─────────────────────────
ws_bc.merge_cells("A30:H30")
sc(
    ws_bc,
    "A30",
    "SECTION 3 — SECTOR GROUP RISK SUMMARY",
    bold=True,
    bg=MID_BLUE,
    font_color=WHITE,
    size=12,
)
ws_bc.row_dimensions[30].height = 28

headers3 = [
    "Sector Group",
    "Total Loans",
    "Default Rate",
    "Avg Loan ($)",
    "Risk Level",
    "Recommendation",
]
for i, h in enumerate(headers3):
    col = chr(65 + i)
    sc(ws_bc, f"{col}31", h, bold=True, bg=DARK_BLUE, font_color=WHITE, size=10)
ws_bc.row_dimensions[31].height = 28

sector_data = [
    ("Food & Beverage", 22185, 6.99, 240000, "HIGH", "Apply strict DTI checks"),
    ("Construction", 17094, 5.00, 290000, "MEDIUM", "Standard screening"),
    ("Services", 9366, 6.25, 135000, "MEDIUM", "Monitor closely"),
    ("Hospitality", 3113, 6.01, 890000, "MEDIUM", "Require collateral"),
    ("Healthcare", 2980, 3.49, 580000, "LOW", "Preferred sector"),
    ("Professional Services", 3262, 3.99, 420000, "LOW", "Preferred sector"),
    ("Transportation", 2750, 7.02, 195000, "HIGH", "Enhanced due diligence"),
    ("Retail", 2380, 5.00, 480000, "MEDIUM", "Standard screening"),
    ("Technology", 1850, 4.50, 650000, "LOW", "Preferred sector"),
    ("Agriculture", 1200, 5.50, 320000, "MEDIUM", "Seasonal risk factor"),
]

for i, (sector, loans, rate, avg, risk, rec) in enumerate(sector_data):
    r = 32 + i
    bg = ind_colors.get(risk, "FFFFFF")
    fc = ind_fc.get(risk, "000000")
    vals = [sector, f"{loans:,}", f"{rate}%", f"${avg:,}", risk, rec]
    for j, (col_letter, val) in enumerate(zip("ABCDEF", vals)):
        sc(
            ws_bc,
            f"{col_letter}{r}",
            val,
            bg=bg,
            font_color=fc if col_letter in ["A", "E"] else "000000",
        )
    ws_bc.row_dimensions[r].height = 22

# ── COLUMN WIDTHS ────────────────────────────────────────────
widths = {"A": 35, "B": 22, "C": 14, "D": 14, "E": 14, "F": 16, "G": 22, "H": 14}
for col, width in widths.items():
    ws_bc.column_dimensions[col].width = width

# ── BAR CHART — Top Industries ───────────────────────────────
chart = BarChart()
chart.type = "bar"
chart.title = "Top 10 Industries by Loan Volume"
chart.y_axis.title = "Industry"
chart.x_axis.title = "Total Loans"
chart.height = 16
chart.width = 20

data = Reference(ws_bc, min_col=3, min_row=12, max_row=22)
cats = Reference(ws_bc, min_col=1, min_row=13, max_row=22)
chart.add_data(data, titles_from_data=True)
chart.set_categories(cats)
chart.series[0].graphicalProperties.solidFill = MID_BLUE

ws_bc.add_chart(chart, "I4")

# ── SAVE ─────────────────────────────────────────────────────
wb6.save("outputs/LoanRisk_Dashboard.xlsm")
print("✅ Business Categories sheet added!")
print("Sections:")
print("  1. Default Rate by Business Type")
print("  2. Top 15 Industries by Loan Volume")
print("  3. Sector Group Risk Summary")
print("  + Bar Chart")

✅ Business Categories sheet added!
Sections:
  1. Default Rate by Business Type
  2. Top 15 Industries by Loan Volume
  3. Sector Group Risk Summary
  + Bar Chart


In [20]:
# === FINAL FIX — WRITE DATA DIRECTLY AND BUILD CHART ===
from openpyxl import load_workbook
from openpyxl.chart import BarChart, Reference
from openpyxl.styles import Font

wb10 = load_workbook("outputs/LoanRisk_Dashboard.xlsm", keep_vba=True)
ws_bc = wb10["Business Categories"]

# Remove all old charts
ws_bc._charts = []

# Write clean data starting at row 45
# Far down so it does not interfere with existing content
industries = [
    ("Full-Service Restaurants", 13017),
    ("Limited-Service Restaurants", 9168),
    ("Landscaping Services", 6250),
    ("Specialty Trade Contractors", 6128),
    ("Residential Remodelers", 6090),
    ("Fitness & Recreation Centers", 5358),
    ("Plumbing & HVAC Contractors", 4876),
    ("Offices of Lawyers", 3262),
    ("Personal Care Services", 3116),
    ("Hotels & Motels", 3113),
]

# Write header
ws_bc["J45"] = "Industry"
ws_bc["K45"] = "Total Loans"
ws_bc["J45"].font = Font(bold=True, name="Arial")
ws_bc["K45"].font = Font(bold=True, name="Arial")

# Write data
for i, (name, loans) in enumerate(industries):
    r = 46 + i
    ws_bc[f"J{r}"] = name
    ws_bc[f"K{r}"] = loans
    ws_bc[f"J{r}"].font = Font(name="Arial", size=9)
    ws_bc[f"K{r}"].font = Font(name="Arial", size=9)

ws_bc.column_dimensions["J"].width = 35
ws_bc.column_dimensions["K"].width = 15

# Build chart from this clean data
chart = BarChart()
chart.type = "bar"
chart.title = "Top 10 Industries by Total Loans"
chart.x_axis.title = "Total Loans"
chart.height = 18
chart.width = 22
chart.style = 10
chart.legend = None

# Data from K46 to K55
data = Reference(ws_bc, min_col=11, min_row=45, max_row=55)

# Categories from J46 to J55
cats = Reference(ws_bc, min_col=10, min_row=46, max_row=55)

chart.add_data(data, titles_from_data=True)
chart.set_categories(cats)
chart.series[0].graphicalProperties.solidFill = "2E75B6"

# Place chart
ws_bc.add_chart(chart, "I4")

wb10.save("outputs/LoanRisk_Dashboard.xlsm")
print("✅ Chart rebuilt with clean data!")
print("Open Excel to check!")

✅ Chart rebuilt with clean data!
Open Excel to check!


In [4]:
# === ADD VBA MACRO CODE ===
# Create a sheet with VBA instructions
ws_vba = wb.create_sheet("VBA Macro Code")
ws_vba.sheet_view.showGridLines = False

vba_instructions = [
    ("HOW TO ADD THE VBA REFRESH BUTTON", DARK_BLUE, WHITE, True, 14),
    ("Step 1: Press Alt + F11 to open VBA Editor", MID_BLUE, WHITE, True, 11),
    ("Step 2: Click Insert → Module", GRAY, "000000", False, 11),
    ("Step 3: Paste the code below into the module", GRAY, "000000", False, 11),
    ("Step 4: Close VBA Editor and go back to Excel", GRAY, "000000", False, 11),
    (
        "Step 5: Click Developer tab → Insert → Button → draw on sheet",
        GRAY,
        "000000",
        False,
        11,
    ),
    ("VBA CODE — COPY AND PASTE INTO VBA EDITOR:", DARK_BLUE, WHITE, True, 11),
]

vba_code = """
Sub RefreshDashboard()

    ' Update timestamp
    Sheets("Executive Summary").Range("A2").Value = _
        "SBA 7(a) Business Loans | 304,590 Records | Last Refreshed: " & _
        Format(Now(), "DD-MMM-YYYY HH:MM")

    ' Recalculate all formulas
    Application.CalculateFull

    ' Autofit all columns on all sheets
    Dim ws As Worksheet
    For Each ws In ThisWorkbook.Worksheets
        If ws.Name <> "VBA Macro Code" Then
            ws.Cells.EntireColumn.AutoFit
        End If
    Next ws

    ' Confirm refresh
    MsgBox "Dashboard refreshed successfully!" & Chr(13) & _
           "Timestamp: " & Format(Now(), "DD-MMM-YYYY HH:MM:SS"), _
           vbInformation, "Refresh Complete"

End Sub

Sub AddRefreshButton()

    Dim btn As Object
    Dim ws As Worksheet
    Set ws = Sheets("Executive Summary")

    Set btn = ws.Buttons.Add(650, 5, 150, 30)
    With btn
        .Caption  = "🔄 Refresh Dashboard"
        .OnAction = "RefreshDashboard"
        .Font.Bold = True
        .Font.Size = 11
        .Font.Name = "Arial"
    End With

    MsgBox "Refresh button added!", vbInformation

End Sub
"""

# Style header rows
for i, (text, bg, fc, bold, size) in enumerate(vba_instructions):
    r = i + 1
    ws_vba.merge_cells(f"A{r}:B{r}")
    style_cell(
        ws_vba, f"A{r}", text, bold=bold, bg=bg, font_color=fc, size=size, align="left"
    )
    ws_vba.row_dimensions[r].height = 28

# Add VBA code lines
start_row = len(vba_instructions) + 1
for i, line in enumerate(vba_code.strip().split("\n")):
    r = start_row + i
    ws_vba.merge_cells(f"A{r}:B{r}")
    c = ws_vba[f"A{r}"]
    c.value = line
    c.font = Font(name="Courier New", size=10, color="000000")
    c.alignment = Alignment(horizontal="left", vertical="center")
    bg = "F5F5F5" if i % 2 == 0 else WHITE
    c.fill = PatternFill("solid", fgColor=bg)
    ws_vba.row_dimensions[r].height = 18

ws_vba.column_dimensions["A"].width = 80
ws_vba.column_dimensions["B"].width = 20

# Save updated file
wb.save("outputs/LoanRisk_Dashboard.xlsx")
print("✅ VBA sheet added and file saved!")
print("Now open the Excel file and follow the VBA instructions!")

✅ VBA sheet added and file saved!
Now open the Excel file and follow the VBA instructions!
